# Feature Engineering & Data Splitting

Before training anything we need to do three things: make sure the model only sees past data, add a few derived signals on top of the raw numbers, and split the data in time order. That's what this notebook is for.

In [39]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [40]:
df = pd.read_csv('../data/cleaned_crisis.csv')
print('Shape:', df.shape)
df.head()

Shape: (1059, 16)


,case,cc3,country,year,systemic_crisis,exch_usd,domestic_debt_in_default,sovereign_external_debt_default,gdp_weighted_default,inflation_annual_cpi,independence,currency_crises,inflation_crises,banking_crisis,inflation_capped,exch_usd_capped
0,1,DZA,Algeria,1870,1,0.052264,0,0,0.0,3.441456,0,0,0,1,3.441456,0.052264
1,1,DZA,Algeria,1871,0,0.052798,0,0,0.0,14.149140,0,0,0,0,14.149140,0.052798
2,1,DZA,Algeria,1872,0,0.052274,0,0,0.0,-3.718593,0,0,0,0,-3.718593,0.052274
3,1,DZA,Algeria,1873,0,0.051680,0,0,0.0,11.203897,0,0,0,0,11.203897,0.051680
4,1,DZA,Algeria,1874,0,0.051308,0,0,0.0,-3.848561,0,0,0,0,-3.848561,0.051308


## 1. Sort and prepare

Sort by country then year. Lags and rolling windows depend on row order within each group, so this has to come first.

In [56]:
df = df.sort_values(['country', 'year']).reset_index(drop=True)

## 2. Lagged features

The model is supposed to predict next year's crisis using only what was known beforehand. So we shift every indicator back by 1 and 2 years. If we used the current year's values, the model would already have the answer.

In [57]:
lag_cols = ['exch_usd_capped', 'inflation_capped',
            'domestic_debt_in_default', 'sovereign_external_debt_default',
            'gdp_weighted_default', 'currency_crises', 'inflation_crises', 'banking_crisis']

# 1‑year lags for each country
for col in lag_cols:
    df[col + '_lag1'] = df.groupby('country')[col].shift(1)

#  2‑year lags
for col in lag_cols:
    df[col + '_lag2'] = df.groupby('country')[col].shift(2)

#### Quick verification of lagged features
- **`_lag1`** : the value from **one year before** the current row.
  Example: for Algeria in 1871, `inflation_lag1` = the inflation of 1870.

- **`_lag2`** : the value from **two years before** the current row.
  Example: for Algeria in 1872, `inflation_lag2` = the inflation of 1870.


In [58]:
lag1_cols = [col + '_lag1' for col in lag_cols]
lag2_cols = [col + '_lag2' for col in lag_cols]

print('New lag columns created:')
print(lag1_cols + lag2_cols)

df[df['country'] == 'Algeria'][['country', 'year'] + lag_cols[:2] +
                                  [lag_cols[0]+'_lag1', lag_cols[1]+'_lag1',
                                   lag_cols[0]+'_lag2', lag_cols[1]+'_lag2']].head(10)

New lag columns created:
['exch_usd_capped_lag1', 'inflation_capped_lag1', 'domestic_debt_in_default_lag1', 'sovereign_external_debt_default_lag1', 'gdp_weighted_default_lag1', 'currency_crises_lag1', 'inflation_crises_lag1', 'banking_crisis_lag1', 'exch_usd_capped_lag2', 'inflation_capped_lag2', 'domestic_debt_in_default_lag2', 'sovereign_external_debt_default_lag2', 'gdp_weighted_default_lag2', 'currency_crises_lag2', 'inflation_crises_lag2', 'banking_crisis_lag2']


,country,year,exch_usd_capped,inflation_capped,exch_usd_capped_lag1,inflation_capped_lag1,exch_usd_capped_lag2,inflation_capped_lag2
0,Algeria,1870,0.052264,3.441456,NaN,NaN,NaN,NaN
1,Algeria,1871,0.052798,14.149140,0.052264,3.441456,NaN,NaN
2,Algeria,1872,0.052274,-3.718593,0.052798,14.149140,0.052264,3.441456
3,Algeria,1873,0.051680,11.203897,0.052274,-3.718593,0.052798,14.149140
4,Algeria,1874,0.051308,-3.848561,0.051680,11.203897,0.052274,-3.718593
5,Algeria,1875,0.051546,-20.924178,0.051308,-3.848561,0.051680,11.203897
6,Algeria,1876,0.051867,-1.769547,0.051546,-20.924178,0.051308,-3.848561
7,Algeria,1877,0.051867,29.116045,0.051867,-1.769547,0.051546,-20.924178
8,Algeria,1878,0.051948,-1.492537,0.051867,29.116045,0.051867,-1.769547
9,Algeria,1879,0.052029,-16.831357,0.051948,-1.492537,0.051867,29.116045


## 3. Exchange Rate Percentage Change

The percentage change tells us **how much** the exchange rate moved compared to the previous year

- **Positive** = the currency **weakened** (depreciation)
- **Negative** = the currency **strengthened** (appreciation)

In [44]:
df['exch_usd_pct_change'] = df.groupby('country')['exch_usd_capped'].pct_change() * 100

#### Verify exchange rate percentage change

In [45]:
df[['country', 'year', 'exch_usd_capped', 'exch_usd_pct_change']].tail(10)

,country,year,exch_usd_capped,exch_usd_pct_change
1049,Zimbabwe,2004,5.600000e-23,6.044025e+02
1050,Zimbabwe,2005,8.460000e-22,1.410714e+03
1051,Zimbabwe,2006,3.000000e-19,3.536099e+04
1052,Zimbabwe,2007,1.900000e-16,6.323333e+04
1053,Zimbabwe,2008,2.000000e-03,1.052632e+15
1054,Zimbabwe,2009,3.548000e+02,1.773990e+07
1055,Zimbabwe,2010,3.782000e+02,6.595265e+00
1056,Zimbabwe,2011,3.619000e+02,-4.309889e+00
1057,Zimbabwe,2012,3.619000e+02,0.000000e+00
1058,Zimbabwe,2013,3.619000e+02,0.000000e+00


## 4. Inflation Volatility

Compute a 3‑year rolling standard deviation of capped inflation.
High volatility often precedes or accompanies economic instability.

In [46]:
df['inflation_volatility'] = df.groupby('country')['inflation_capped'] \
                                .rolling(window=3, min_periods=3).std() \
                                .reset_index(level=0, drop=True)

#### Verify inflation volatility

In [47]:
df[df['country'] == 'Algeria'][['country', 'year', 'inflation_capped', 'inflation_volatility']].head(8)

,country,year,inflation_capped,inflation_volatility
0,Algeria,1870,3.441456,NaN
1,Algeria,1871,14.149140,NaN
2,Algeria,1872,-3.718593,8.992373
3,Algeria,1873,11.203897,9.579588
4,Algeria,1874,-3.848561,8.653266
5,Algeria,1875,-20.924178,16.074651
6,Algeria,1876,-1.769547,10.510304
7,Algeria,1877,29.116045,25.248246


## 5. Create the Target: Next Year’s Systemic Crisis

We shift `systemic_crisis` **backward** by one year:
for each row of year *t*, the target `crisis_next_year` tells us whether a crisis happened in *t+1*.

In [48]:
df['crisis_next_year'] = df.groupby('country')['systemic_crisis'].shift(-1)

df = df.dropna(subset=['crisis_next_year']).reset_index(drop=True)

print('Rows after target shifting:', df.shape[0])

Rows after target shifting: 1046


In [61]:
lag_feature_cols = [col + '_lag1' for col in lag_cols] + [col + '_lag2' for col in lag_cols]
df = df.dropna(subset=lag_feature_cols)
print('Rows after dropping lag NaNs:', df.shape[0])

Rows after dropping lag NaNs: 1020


## 6. Time‑Aware Train / Validation / Test Split

We split **chronologically** to simulate a realistic forecasting scenario:
- **Train** : all years ≤ 1999
- **Validation** : 2000 – 2009
- **Test** : 2010 – 2014

No future information leaks into training.

In [49]:
train = df[df['year'] <= 1999]
val   = df[(df['year'] >= 2000) & (df['year'] <= 2009)]
test  = df[df['year'] >= 2010]

print(f'Train: {train.shape[0]} rows, {train["year"].min()}–{train["year"].max()}')
print(f'Val:   {val.shape[0]} rows, {val["year"].min()}–{val["year"].max()}')
print(f'Test:  {test.shape[0]} rows, {test["year"].min()}–{test["year"].max()}')

Train: 866 rows, 1860–1999
Val:   130 rows, 2000–2009
Test:  50 rows, 2010–2013


## 7. Check Target Class Balance

The minority class (crisis) is rare – we verify each split still contains crisis examples.

In [59]:
for name, split in [('Train', train), ('Validation', val), ('Test', test)]:
    count_1 = split['crisis_next_year'].sum()
    total = len(split)
    print(f'{name}: {count_1} crises out of {total} ({count_1/total:.2%})')

Train: 66.0 crises out of 866 (7.62%)
Validation: 11.0 crises out of 130 (8.46%)
Test: 4.0 crises out of 50 (8.00%)


In [60]:
# Verify that years for each country are strictly separated
for country in df['country'].unique():
    train_years = train[train['country']==country]['year']
    val_years   = val[val['country']==country]['year']
    test_years  = test[test['country']==country]['year']
    if len(train_years)>0 and len(val_years)>0 and len(test_years)>0:
        assert train_years.max() < val_years.min(), f'Overlap in {country}'
        assert val_years.max() < test_years.min(), f'Overlap in {country}'
print('No data leakage detected.')

No data leakage detected.


## 8. Save the Processed Datasets

In [67]:
train.to_csv('../data/train.csv', index=False)
val.to_csv('../data/validation.csv', index=False)
test.to_csv('../data/test.csv', index=False)

print('Train, validation, and test sets saved to ../data/')

Train, validation, and test sets saved to ../data/


### Summary of Feature Engineering
- **Lagged features:** 1‑ and 2‑year lags of economic indicators and crisis flags.
- **Derived features:** Exchange rate % change, 3‑year inflation volatility.
- **Target:** `crisis_next_year` – systemic crisis in year *t+1*.
- **Time‑aware split:** Train ≤1999, Val 2000‑2009, Test 2010‑2014 - no future leakage.